In [ ]:
import random
import json
import mysql.connector
from faker import Faker

# =====================================================
# CONFIGURATION
# =====================================================

TOTAL_CUSTOMERS = 500

random.seed(100)
fake = Faker("en_IN")
Faker.seed(100)

# =====================================================
# MYSQL CONNECTION
# =====================================================

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="mani@sql5825u",
    database="history_builder"
)

cursor = conn.cursor()

# =====================================================
# MASTER DATA
# =====================================================

cities = [
    "Chennai",
    "Coimbatore",
    "Madurai",
    "Trichy",
    "Salem",
    "Erode",
    "Tiruppur",
    "Bangalore",
    "Hyderabad",
    "Mumbai",
    "Pune",
    "Kochi"
]

city_weights = [
    20, 15, 8, 6, 5, 4, 4,
    12, 8, 8, 5, 5
]

devices = [
    "Samsung S24",
    "Samsung A55",
    "Samsung M35",
    "Redmi Note 13",
    "Redmi Note 14",
    "OnePlus 13",
    "Vivo V50",
    "iPhone 15",
    "iPhone 16"
]

device_weights = [
    12,
    18,
    15,
    18,
    10,
    10,
    7,
    5,
    5
]

payment_methods = [
    "UPI",
    "Debit Card",
    "Credit Card",
    "Net Banking"
]

payment_weights = [
    65,
    18,
    12,
    5
]

# =====================================================
# CUSTOMER TYPES
# =====================================================

customer_types = [

    {
        "type":"Student",
        "weight":20,
        "amount":(100,1200),
        "activity":(1,4)
    },

    {
        "type":"Salaried",
        "weight":45,
        "amount":(300,3500),
        "activity":(2,6)
    },

    {
        "type":"Business",
        "weight":20,
        "amount":(2000,15000),
        "activity":(5,12)
    },

    {
        "type":"Premium",
        "weight":10,
        "amount":(5000,40000),
        "activity":(2,8)
    },

    {
        "type":"Retired",
        "weight":5,
        "amount":(200,2500),
        "activity":(1,3)
    }

]

customer_pool = []

for profile in customer_types:
    customer_pool.extend([profile] * profile["weight"])

# =====================================================
# SQL
# =====================================================

insert_query = """
INSERT INTO customers
(
customer_name,
account_number,
phone_number,
home_city,
usual_device,
usual_payment_method
)
VALUES
(%s,%s,%s,%s,%s,%s)
"""

used_accounts = set()
used_phones = set()

customer_profiles = {}

print("Generating Customers...\n")

# =====================================================
# CUSTOMER GENERATION
# =====================================================

for _ in range(TOTAL_CUSTOMERS):

    while True:
        account = str(random.randint(100000000000,999999999999))
        if account not in used_accounts:
            used_accounts.add(account)
            break

    while True:
        phone = str(random.randint(6000000000,9999999999))
        if phone not in used_phones:
            used_phones.add(phone)
            break

    city = random.choices(
        cities,
        weights=city_weights,
        k=1
    )[0]

    device = random.choices(
        devices,
        weights=device_weights,
        k=1
    )[0]

    payment = random.choices(
        payment_methods,
        weights=payment_weights,
        k=1
    )[0]

    cursor.execute(
        insert_query,
        (
            fake.name(),
            account,
            phone,
            city,
            device,
            payment
        )
    )

    customer_id = cursor.lastrowid

    profile = random.choice(customer_pool)

    avg_amount = random.randint(
        profile["amount"][0],
        profile["amount"][1]
    )

    customer_profiles[str(customer_id)] = {

        "customer_type": profile["type"],

        "average_amount": avg_amount,

        "variation": round(
            random.uniform(0.12,0.30),
            2
        ),

        "transactions_per_day": random.randint(
            profile["activity"][0],
            profile["activity"][1]
        ),

        "active_start": random.randint(6,10),

        "active_end": random.randint(19,23),

        "weekend_multiplier": round(
            random.uniform(1.10,1.40),
            2
        ),

        "salary_day": random.randint(1,7),

        "travel_probability": round(
            random.uniform(0.01,0.03),
            3
        ),

        "device_change_probability": round(
            random.uniform(0.01,0.02),
            3
        ),

        "payment_change_probability": round(
            random.uniform(0.05,0.12),
            3
        )

    }

conn.commit()

with open("model/customer_profiles.json","w") as f:
    json.dump(customer_profiles,f,indent=4)

print(f"{TOTAL_CUSTOMERS} Customers Generated Successfully.")

cursor.close()
conn.close()

Generating Customers...

500 Customers Generated Successfully.


In [ ]:
import json
import random
from datetime import datetime, timedelta
import mysql.connector



START_DATE = datetime(2025, 1, 1)
END_DATE = datetime(2025, 6, 30)

TOTAL_TARGET_TRANSACTIONS = 30000

random.seed(100)



conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="mani@sql5825u",
    database="history_builder"
)

cursor = conn.cursor(dictionary=True)



cursor.execute("SELECT * FROM customers")
customers = cursor.fetchall()

print(f"Loaded Customers: {len(customers)}")



with open("model/customer_profiles.json", "r") as f:
    customer_profiles = json.load(f)

print(f"Loaded Profiles: {len(customer_profiles)}")


transaction_log = []
daily_customer_map = {}

for c in customers:
    daily_customer_map[c["customer_id"]] = []

print("Base structures initialized.")

Loaded Customers: 500
Loaded Profiles: 500
Base structures initialized.


In [ ]:
import random
from datetime import timedelta



def is_active_today(profile):

    base_activity = profile["transactions_per_day"]

    
    if base_activity <= 1:
        return random.random() < 0.35
    elif base_activity <= 3:
        return random.random() < 0.60
    elif base_activity <= 6:
        return random.random() < 0.80
    else:
        return random.random() < 0.95




def get_daily_tx_count(profile):

    base = profile["transactions_per_day"]

    
    variation = random.randint(-1, 2)

    count = max(0, base + variation)

   
    return count




def generate_days(start, end):

    days = (end - start).days

    for i in range(days):
        yield start + timedelta(days=i)




def generate_time(profile, day):

    start_hour = profile["active_start"]
    end_hour = profile["active_end"]

    peak_hours = [9, 13, 18, 20]

    if random.random() < 0.6:
        hour = random.choice(peak_hours)
    else:
        hour = random.randint(start_hour, end_hour)

    minute = random.randint(0, 59)
    second = random.randint(0, 59)

    return day.replace(hour=hour, minute=minute, second=second)





def is_weekend(day):

    return day.weekday() in [5, 6]

In [ ]:
import random



def generate_amount(profile, is_weekend=False):

    base = profile["average_amount"]
    variation = profile["variation"]

   
    amount = random.uniform(
        base * (1 - variation),
        base * (1 + variation)
    )

  
    if is_weekend:
        amount *= profile["weekend_multiplier"]

  
    if random.random() < 0.02:
        amount *= random.uniform(2, 4)

    return round(max(10, amount), 2)



def generate_normal_transaction(customer, profile, day):

    is_weekend_flag = is_weekend(day)

    
    amount = generate_amount(profile, is_weekend_flag)

   
   
    
    if random.random() < 0.96:
        device = customer["usual_device"]
    else:
        device = random.choice(devices)

 
   
    if random.random() < 0.98:
        city = customer["home_city"]
    else:
        city = random.choice(cities)

    
    
    if random.random() < 0.92:
        payment = customer["usual_payment_method"]
    else:
        payment = random.choice(payment_methods)

    
    tx_time = generate_time(profile, day)

    return (
        customer["customer_id"],
        tx_time,
        amount,
        payment,
        city,
        device,
        None   
    )

In [ ]:
import random



transactions_to_insert = []



print("\nStarting History Simulation...\n")

total_generated = 0

for day in generate_days(START_DATE, END_DATE):

    for customer in customers:

        customer_id = customer["customer_id"]
        profile = customer_profiles[str(customer_id)]

       
        if not is_active_today(profile):
            continue

       
        tx_count = get_daily_tx_count(profile)

        
        for _ in range(tx_count):

            tx = generate_normal_transaction(
                customer,
                profile,
                day
            )

            transactions_to_insert.append(tx)

            total_generated += 1

  
    if total_generated % 2000 == 0:
        print(f"Generated: {total_generated}")

print("\nSimulation Complete")
print("Total Transactions Generated:", total_generated)


Starting History Simulation...


Simulation Complete
Total Transactions Generated: 361562


In [ ]:
import numpy as np
import pandas as pd


insert_query = """
INSERT INTO transactions
(
    customer_id,
    transaction_time,
    amount,
    payment_method,
    city,
    device,
    fraud_label
)
VALUES (%s,%s,%s,%s,%s,%s,%s)
"""

print("\nInserting into MySQL...\n")

BATCH_SIZE = 2000

for i in range(0, len(transactions_to_insert), BATCH_SIZE):

    batch = transactions_to_insert[i:i+BATCH_SIZE]

    cursor.executemany(insert_query, batch)
    conn.commit()

    print(f"Inserted: {i + len(batch)} / {len(transactions_to_insert)}")

print("\nAll transactions inserted successfully.")



df = pd.DataFrame(
    transactions_to_insert,
    columns=[
        "customer_id",
        "transaction_time",
        "amount",
        "payment_method",
        "city",
        "device",
        "fraud_label"
    ]
)



print("\n===== DATA DISTRIBUTION CHECK =====\n")

print("Total Transactions:", len(df))

print("\nAmount Stats:")
print(df["amount"].describe())

print("\nPayment Method Distribution:")
print(df["payment_method"].value_counts(normalize=True))

print("\nCity Distribution (Top 5):")
print(df["city"].value_counts().head())

print("\nDevice Distribution (Top 5):")
print(df["device"].value_counts().head())

print("\nTransactions per Customer:")
print(df["customer_id"].value_counts().describe())


df["hour"] = pd.to_datetime(df["transaction_time"]).dt.hour

print("\nHour Distribution:")
print(df["hour"].value_counts().sort_index())



print("\n HISTORY BUILDER COMPLETED SUCCESSFULLY ")
print("Dataset is now realistic and ready for fraud detection validation.")


Inserting into MySQL...

Inserted: 2000 / 361562
Inserted: 4000 / 361562
Inserted: 6000 / 361562
Inserted: 8000 / 361562
Inserted: 10000 / 361562
Inserted: 12000 / 361562
Inserted: 14000 / 361562
Inserted: 16000 / 361562
Inserted: 18000 / 361562
Inserted: 20000 / 361562
Inserted: 22000 / 361562
Inserted: 24000 / 361562
Inserted: 26000 / 361562
Inserted: 28000 / 361562
Inserted: 30000 / 361562
Inserted: 32000 / 361562
Inserted: 34000 / 361562
Inserted: 36000 / 361562
Inserted: 38000 / 361562
Inserted: 40000 / 361562
Inserted: 42000 / 361562
Inserted: 44000 / 361562
Inserted: 46000 / 361562
Inserted: 48000 / 361562
Inserted: 50000 / 361562
Inserted: 52000 / 361562
Inserted: 54000 / 361562
Inserted: 56000 / 361562
Inserted: 58000 / 361562
Inserted: 60000 / 361562
Inserted: 62000 / 361562
Inserted: 64000 / 361562
Inserted: 66000 / 361562
Inserted: 68000 / 361562
Inserted: 70000 / 361562
Inserted: 72000 / 361562
Inserted: 74000 / 361562
Inserted: 76000 / 361562
Inserted: 78000 / 361562
Ins